# 0 — Data and weights

Everything the tutorials read lives under `data/` and `checkpoints/`. Released under [CC BY-NC-ND 4.0](https://creativecommons.org/licenses/by-nc-nd/4.0/).

| what | tutorials | size | repo |
|---|---|---|---|
| crc-metu | 6 | 73 MB | `yandrewl/QUEST-tutorial-data` |
| bog-86337 | 4 | 36 MB | `yandrewl/QUEST-tutorial-data` |
| stanford-pc | 1, 5 | 1.6 GB | `yandrewl/QUEST-tutorial-data` |
| weights | all | 1.5 GB | `yandrewl/QUEST` |
| Eva encoder | 5 | 490 MB | [`yandrewl/Eva`](https://huggingface.co/yandrewl/Eva) |
| UNI2-h | all | — | [`MahmoodLab/UNI2-h`](https://huggingface.co/MahmoodLab/UNI2-h) |
| PathoCell | 2, 3 | ~350 MB / region | [`Kainmueller-Lab/PathoCell`](https://huggingface.co/datasets/Kainmueller-Lab/PathoCell) |

## Weights

```
checkpoints/
  quest_semantic.ckpt   marker queries from a semantic embedding of the marker's name
  quest_id.ckpt         the same architecture with a learned per-marker table instead
  quest_eva.ckpt        a masked autoencoder that inpaints a partly observed stack
  celltyper_quest-semantic.npz   the cell typer of tutorials 2 and 3
  eva_marker_table.npz  Eva's marker query, 129 x 3072, read by questkit.retrieval.EvaEmbedder
  Eva_model.ckpt        yandrewl/Eva
```

Each QUEST checkpoint carries its own marker table, so it needs no marker-embedding file;
`Eva_model.ckpt` does not, which is what `eva_marker_table.npz` is for.

## Download

`huggingface-cli login` first.

```python
import shutil
from huggingface_hub import snapshot_download, hf_hub_download

snapshot_download("yandrewl/QUEST", local_dir="checkpoints",
                  allow_patterns=["*.ckpt", "*.npz"])

snapshot_download("yandrewl/QUEST-tutorial-data", repo_type="dataset", local_dir="data",
                  allow_patterns=["crc-metu/*",      # tutorial 6
                                  "bog-86337/*",     # tutorial 4
                                  "stanford-pc/*"])  # tutorials 1, 5

shutil.copy(hf_hub_download("yandrewl/Eva", "Eva_model.ckpt"),  
            "checkpoints/Eva_model.ckpt")                       
from questkit import cohort
cohort.fetch_pathocell(["reg016_B", "reg032_B"])
```

## crc-metu

A held-out colorectal cohort: **57 patients, 37 of whom died**, contributing 62 whole-slide
regions that carry `survival_metU`. Four patients contributed more than one region, which is why
there are 62 rows for 57 patients.

In [1]:
import _common  # noqa: F401
import csv, json, os
import numpy as np
from pathlib import Path

REPO = Path(_common.QUEST_CODE)
DATA = Path(os.environ.get("QUEST_DATA") or REPO / "data")

def mb(p):
    p = Path(p)
    if not p.exists(): return None
    return (p.stat().st_size if p.is_file()
            else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())) / 1e6

for name, nbs in (("crc-metu", "6"), ("stanford-pc", "1, 5"), ("bog-86337", "4"),
                  ("PathoCell", "2, 3")):
    s = mb(DATA / name)
    print(f"  {name:14s} tutorials {nbs:6s} " + (f"{s:9.1f} MB" if s else "        MISSING"))
for f in ("quest_semantic.ckpt", "quest_id.ckpt", "quest_eva.ckpt",
          "celltyper_quest-semantic.npz", "eva_marker_table.npz", "Eva_model.ckpt"):
    s = mb(REPO / "checkpoints" / f)
    print(f"  checkpoints/{f:32s} " + (f"{s:9.1f} MB" if s else "  MISSING"))

  crc-metu       tutorials 6           72.9 MB
  stanford-pc    tutorials 1, 5      1625.6 MB
  bog-86337      tutorials 4           35.6 MB
  PathoCell      tutorials 2, 3     38273.6 MB
  checkpoints/quest_semantic.ckpt                  511.0 MB
  checkpoints/quest_id.ckpt                        513.6 MB
  checkpoints/quest_eva.ckpt                       491.4 MB
  checkpoints/celltyper_quest-semantic.npz           0.0 MB
  checkpoints/eva_marker_table.npz                   1.3 MB
  checkpoints/Eva_model.ckpt                       489.9 MB


In [2]:
R = DATA / "crc-metu"
if not R.exists():
    print("crc-metu not present")
else:
    files = sorted((R / "feats").glob("*.npz"))
    with np.load(files[0], allow_pickle=True) as z:
        print(f"feats/  {len(files)} files")
        for k in z.files:
            print(f"    {k:12s} {str(z[k].shape):14s} {z[k].dtype}")
        print("    panel:", ", ".join(str(m) for m in z["panel"]))

    idx = json.loads((R / "slide_index.json").read_text())
    rows = list(csv.DictReader(open(R / "labels.csv")))
    print(f"\nslide_index.json  {len(idx)} x {list(idx[0])}")
    print(f"labels.csv        {len(rows)} x {list(rows[0])}")
    print(f"                  {len({r['patient_id'] for r in rows})} patients, "
          f"{sum(r['os_event_metU'] == '1' for r in rows)} events")

    for p in sorted((R / "cases").glob("*.npz")):
        with np.load(p) as z:
            print(f"\ncases/{p.name}")
            for k in z.files:
                print(f"    {k:14s} {str(z[k].shape):18s} {z[k].dtype}")

    o = json.loads((R / "mil_oof_metU.json").read_text())
    print(f"\nmil_oof_metU.json  n={o['n']} events={o['events']} folds={o['folds']} "
          f"seed={o['seed']}  cindex_virt {o['cindex_virt']:.3f}")
    print(f"    patients[{next(iter(o['patients']))}]: "
          f"{list(next(iter(o['patients'].values())))}")

feats/  62 files
    dino_vmif    (64, 8, 768)   float16
    slide_idx    ()             int64
    patient_id   ()             <U7
    panel        (8,)           <U5
    eva_norm     ()             <U3
    panel: DAPI, CD3e, CD8, CD20, CD68, Ki67, PDL1, PanCK

slide_index.json  62 x ['slide_idx', 'patient_id']
labels.csv        62 x ['patient_id', 'survival_metU', 'os_event_metU']
                  57 patients, 42 events

cases/69T_1.npz
    he             (59, 112, 112, 3)  uint8
    coords         (59, 2)            int64
    patient_id     ()                 <U5
    acquisition_id ()                 <U9
    stride         ()                 int64
    downsample     ()                 int64

mil_oof_metU.json  n=57 events=37 folds=5 seed=0  cindex_virt 0.711
    patients[CX-42_2]: ['time', 'event', 'n_slides', 'n_tiles', 'risk_virt', 'loo_virt']
